# Bayesian Optimization Competition

This notebook implements a systematic approach to optimizing 8 black-box functions using Bayesian optimization with surrogate models.

## Competition Overview

- **Objective**: Maximize 8 black-box functions with different dimensionalities (2D to 8D)
- **Constraint**: 1 query per function per week
- **Approach**: Use surrogate models to predict function behavior and select optimal next queries

## Notebook Structure

1. **Setup and Data Loading** - Initialize environment and load function data
2. **Surrogate Model Framework** - Modular framework for different surrogate models
3. **Acquisition Functions** - Functions to select next query points
4. **Function Analysis Dashboard** - Visualize and analyze each function
5. **Weekly Query Generator** - Generate recommended queries for submission
6. **Update Models with Results** - Add new observations and retrain
7. **Progress Tracking** - Monitor optimization progress over time

# Section 1: Setup and Data Loading

In [19]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from abc import ABC, abstractmethod
from typing import Tuple, Dict, List, Optional
import warnings
warnings.filterwarnings('ignore')

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from scipy.stats import norm
from scipy.optimize import minimize

# Set random seed for reproducibility
np.random.seed(42)

# Configure matplotlib
plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("✓ All libraries imported successfully")

✓ All libraries imported successfully


## FunctionData Class

In [20]:
class FunctionData:
    """Manages data for a single black-box function."""
    
    def __init__(self, function_id: int, data_dir: str = "../data"):
        self.function_id = function_id
        self.data_dir = Path(data_dir)
        self.function_dir = self.data_dir / f"function_{function_id}"
        
        # Load initial data
        self.inputs = np.load(self.function_dir / "initial_inputs.npy")
        self.outputs = np.load(self.function_dir / "initial_outputs.npy")
        
        # Store dimensions
        self.n_samples = len(self.outputs)
        self.n_dims = self.inputs.shape[1]
        
        # Track weekly updates
        self.week_number = 0
        self.history = []
        
    def add_observation(self, x: np.ndarray, y: float, week: Optional[int] = None):
        if week is None:
            self.week_number += 1
            week = self.week_number
        
        self.inputs = np.vstack([self.inputs, x.reshape(1, -1)])
        self.outputs = np.append(self.outputs, y)
        self.n_samples += 1
        self.history.append((week, x.copy(), y))
        
    def get_best(self) -> Tuple[np.ndarray, float]:
        best_idx = np.argmax(self.outputs)
        return self.inputs[best_idx], self.outputs[best_idx]
    
    def save_weekly_data(self, week: int):
        week_file_inputs = self.function_dir / f"week_{week}_inputs.npy"
        week_file_outputs = self.function_dir / f"week_{week}_outputs.npy"
        np.save(week_file_inputs, self.inputs)
        np.save(week_file_outputs, self.outputs)
        
    def get_summary(self) -> Dict:
        return {
            'function_id': self.function_id,
            'n_dims': self.n_dims,
            'n_samples': self.n_samples,
            'best_value': np.max(self.outputs),
            'mean_value': np.mean(self.outputs),
            'std_value': np.std(self.outputs)
        }
    
    def __repr__(self):
        best_x, best_y = self.get_best()
        return f"Function {self.function_id}: {self.n_dims}D, {self.n_samples} samples, best={best_y:.6f}"

print("✓ FunctionData class defined")

✓ FunctionData class defined


## Load All Functions

In [21]:
# Load all 8 functions
functions = {}
for i in range(1, 9):
    try:
        functions[i] = FunctionData(i)
        print(f"✓ Loaded {functions[i]}")
    except Exception as e:
        print(f"✗ Error loading function {i}: {e}")

print(f"\n✓ Successfully loaded {len(functions)} functions")

✓ Loaded Function 1: 2D, 10 samples, best=0.000000
✓ Loaded Function 2: 2D, 10 samples, best=0.611205
✓ Loaded Function 3: 3D, 15 samples, best=-0.034835
✓ Loaded Function 4: 4D, 30 samples, best=-4.025542
✓ Loaded Function 5: 4D, 20 samples, best=1088.859618
✓ Loaded Function 6: 5D, 20 samples, best=-0.714265
✓ Loaded Function 7: 6D, 30 samples, best=1.364968
✓ Loaded Function 8: 8D, 40 samples, best=9.598482

✓ Successfully loaded 8 functions


## Display Summary Statistics

In [22]:
print("=" * 80)
print(f"{'Func':<6} {'Dims':<6} {'Samples':<8} {'Best':<12} {'Mean':<12} {'Std':<12}")
print("=" * 80)

for func_id, func_data in functions.items():
    summary = func_data.get_summary()
    print(f"{func_id:<6} {summary['n_dims']:<6} {summary['n_samples']:<8} "
          f"{summary['best_value']:<12.6f} {summary['mean_value']:<12.6f} "
          f"{summary['std_value']:<12.6f}")

print("=" * 80)

Func   Dims   Samples  Best         Mean         Std         
1      2      10       0.000000     -0.000361    0.001082    
2      2      10       0.611205     0.230674     0.225365    
3      3      15       -0.034835    -0.107167    0.084214    
4      4      30       -4.025542    -17.238587   7.017985    
5      4      20       1088.859618  151.271876   245.575981  
6      5      20       -0.714265    -1.495390    0.449000    
7      6      30       1.364968     0.219607     0.302129    
8      8      40       9.598482     7.815274     0.946903    


# Section 2: Surrogate Model Framework

In [23]:
class SurrogateModel(ABC):
    """Abstract base class for surrogate models."""
    
    @abstractmethod
    def fit(self, X: np.ndarray, y: np.ndarray):
        pass
    
    @abstractmethod
    def predict(self, X: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        pass
    
    @abstractmethod
    def get_name(self) -> str:
        pass

print("✓ SurrogateModel base class defined")

✓ SurrogateModel base class defined


## Gaussian Process Surrogate

In [24]:
class GPSurrogate(SurrogateModel):
    """Gaussian Process surrogate model with RBF kernel."""
    
    def __init__(self, length_scale: float = 1.0, length_scale_bounds: Tuple = (1e-2, 1e2),
                 noise: float = 1e-5, optimize: bool = True):
        self.length_scale = length_scale
        self.length_scale_bounds = length_scale_bounds
        self.noise = noise
        self.optimize = optimize
        
        # Create kernel
        kernel = C(1.0, (1e-3, 1e3)) * RBF(length_scale, length_scale_bounds)
        
        # Create GP model
        self.model = GaussianProcessRegressor(
            kernel=kernel,
            alpha=noise,
            n_restarts_optimizer=10 if optimize else 0,
            normalize_y=True,
            random_state=42
        )
        
        self.is_fitted = False
        
    def fit(self, X: np.ndarray, y: np.ndarray):
        self.model.fit(X, y)
        self.is_fitted = True
        
    def predict(self, X: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        if not self.is_fitted:
            raise ValueError("Model must be fitted before prediction")
        mean, std = self.model.predict(X, return_std=True)
        return mean, std
    
    def get_name(self) -> str:
        return f"GP-RBF (ls={self.length_scale:.3f})"

print("✓ GPSurrogate class defined")

✓ GPSurrogate class defined


# Section 3: Acquisition Functions

In [25]:
class AcquisitionFunction:
    """Collection of acquisition functions for Bayesian optimization."""
    
    @staticmethod
    def ucb(mean: np.ndarray, std: np.ndarray, beta: float = 2.0) -> np.ndarray:
        """Upper Confidence Bound: UCB = mean + beta * std"""
        return mean + beta * std
    
    @staticmethod
    def ei(mean: np.ndarray, std: np.ndarray, y_best: float, xi: float = 0.01) -> np.ndarray:
        """Expected Improvement"""
        std = np.maximum(std, 1e-9)
        z = (mean - y_best - xi) / std
        ei_value = (mean - y_best - xi) * norm.cdf(z) + std * norm.pdf(z)
        return ei_value
    
    @staticmethod
    def pi(mean: np.ndarray, std: np.ndarray, y_best: float, xi: float = 0.01) -> np.ndarray:
        """Probability of Improvement"""
        std = np.maximum(std, 1e-9)
        z = (mean - y_best - xi) / std
        pi_value = norm.cdf(z)
        return pi_value

print("✓ Acquisition functions defined (UCB, EI, PI)")

✓ Acquisition functions defined (UCB, EI, PI)


## Optimize Acquisition Function

In [26]:
def optimize_acquisition(surrogate: SurrogateModel, func_data: FunctionData,
                         acq_func: str = 'ucb', n_random: int = 1000,
                         n_refine: int = 10, **acq_params) -> np.ndarray:
    """Find the point that maximizes the acquisition function."""
    n_dims = func_data.n_dims
    
    # Determine bounds from existing data
    bounds = []
    for i in range(n_dims):
        x_min, x_max = func_data.inputs[:, i].min(), func_data.inputs[:, i].max()
        margin = (x_max - x_min) * 0.1
        bounds.append((max(0, x_min - margin), min(1, x_max + margin)))
    
    # Stage 1: Random sampling
    X_random = np.random.uniform(0, 1, size=(n_random, n_dims))
    for i in range(n_dims):
        X_random[:, i] = X_random[:, i] * (bounds[i][1] - bounds[i][0]) + bounds[i][0]
    
    mean, std = surrogate.predict(X_random)
    
    # Calculate acquisition values
    if acq_func == 'ucb':
        acq_values = AcquisitionFunction.ucb(mean, std, beta=acq_params.get('beta', 2.0))
    elif acq_func == 'ei':
        _, y_best = func_data.get_best()
        acq_values = AcquisitionFunction.ei(mean, std, y_best, xi=acq_params.get('xi', 0.01))
    elif acq_func == 'pi':
        _, y_best = func_data.get_best()
        acq_values = AcquisitionFunction.pi(mean, std, y_best, xi=acq_params.get('xi', 0.01))
    else:
        raise ValueError(f"Unknown acquisition function: {acq_func}")
    
    # Stage 2: Local refinement
    best_candidates_idx = np.argsort(acq_values)[-n_refine:]
    best_candidate = None
    best_acq_value = -np.inf
    
    for idx in best_candidates_idx:
        x0 = X_random[idx]
        
        def objective(x):
            x = x.reshape(1, -1)
            mean, std = surrogate.predict(x)
            if acq_func == 'ucb':
                return -AcquisitionFunction.ucb(mean, std, beta=acq_params.get('beta', 2.0))[0]
            elif acq_func == 'ei':
                _, y_best = func_data.get_best()
                return -AcquisitionFunction.ei(mean, std, y_best, xi=acq_params.get('xi', 0.01))[0]
            else:
                _, y_best = func_data.get_best()
                return -AcquisitionFunction.pi(mean, std, y_best, xi=acq_params.get('xi', 0.01))[0]
        
        result = minimize(objective, x0, method='L-BFGS-B', bounds=bounds)
        
        if -result.fun > best_acq_value:
            best_acq_value = -result.fun
            best_candidate = result.x
    
    return best_candidate

print("✓ Acquisition optimization function defined")

✓ Acquisition optimization function defined


# Section 4: Function Analysis

Analyze individual functions with visualization (for 2D functions).

In [27]:
def analyze_function(func_id: int, surrogate: SurrogateModel = None, 
                     acq_func: str = 'ucb', **acq_params):
    """Complete analysis of a single function."""
    func_data = functions[func_id]
    
    # Create and fit surrogate if not provided
    if surrogate is None:
        surrogate = GPSurrogate(length_scale=0.5, optimize=True)
    
    surrogate.fit(func_data.inputs, func_data.outputs)
    
    # Find next query point
    next_query = optimize_acquisition(surrogate, func_data, acq_func=acq_func, **acq_params)
    
    # Display summary
    best_x, best_y = func_data.get_best()
    print("=" * 80)
    print(f"FUNCTION {func_id} ANALYSIS")
    print("=" * 80)
    print(f"Dimensions: {func_data.n_dims}D")
    print(f"Samples: {func_data.n_samples}")
    print(f"Best observed value: {best_y:.6f}")
    print(f"Best input: {np.array2string(best_x, precision=4, separator=', ')}")
    print(f"\\nSurrogate model: {surrogate.get_name()}")
    print(f"Acquisition function: {acq_func.upper()}")
    print(f"\\nSuggested next query:")
    print(f"  x = {np.array2string(next_query, precision=6, separator=', ')}")
    
    # Predict at next query
    mean, std = surrogate.predict(next_query.reshape(1, -1))
    print(f"  Predicted: mean={mean[0]:.6f}, std={std[0]:.6f}")
    print("=" * 80)
    print()
    
    # Visualize for 2D functions
    if func_data.n_dims == 2:
        # Simple visualization for 2D
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.scatter(func_data.inputs[:, 0], func_data.inputs[:, 1], 
                  c=func_data.outputs, s=100, cmap='viridis', edgecolors='white', linewidths=2)
        ax.scatter(next_query[0], next_query[1], c='red', s=300, marker='*', 
                  edgecolors='white', linewidths=2, label='Next Query')
        ax.set_xlabel('x₀')
        ax.set_ylabel('x₁')
        ax.set_title(f'Function {func_id}: Current Observations and Next Query')
        ax.legend()
        plt.colorbar(ax.collections[0], ax=ax, label='Function Value')
        plt.show()
    else:
        print(f"(2D visualization skipped for {func_data.n_dims}D function)")
    
    return next_query, surrogate

print("✓ Analysis function defined")

✓ Analysis function defined


In [28]:
## Example: Analyze a Specific Function

#Change the function_id to analyze different functions (1-8).

# Analyze function 1 (2D function with visualization)
next_query, surrogate_model = analyze_function(6, acq_func='ucb', beta=2.5)

FUNCTION 6 ANALYSIS
Dimensions: 5D
Samples: 20
Best observed value: -0.714265
Best input: [0.7282, 0.1547, 0.7326, 0.694 , 0.0564]
\nSurrogate model: GP-RBF (ls=0.500)
Acquisition function: UCB
\nSuggested next query:
  x = [0.248237, 0.314846, 0.411998, 0.758207, 0.      ]
  Predicted: mean=-0.664995, std=0.368419

(2D visualization skipped for 5D function)


# Section 5: Weekly Query Generator

Generate query points for all 8 functions at once for weekly submission.

In [29]:
def generate_weekly_queries(acq_func: str = 'ucb', surrogate_params: Dict = None,
                           **acq_params) -> Dict[int, np.ndarray]:
    """Generate query points for all functions."""
    if surrogate_params is None:
        surrogate_params = {'length_scale': 0.5, 'optimize': True}
    
    queries = {}
    surrogates = {}
    
    print("=" * 80)
    print("GENERATING WEEKLY QUERIES FOR ALL FUNCTIONS")
    print("=" * 80)
    print(f"Acquisition function: {acq_func.upper()}")
    print(f"Acquisition params: {acq_params}")
    print("=" * 80)
    print()
    
    for func_id in range(1, 9):
        func_data = functions[func_id]
        
        # Create and fit surrogate
        surrogate = GPSurrogate(**surrogate_params)
        surrogate.fit(func_data.inputs, func_data.outputs)
        
        # Find next query
        next_query = optimize_acquisition(surrogate, func_data, acq_func=acq_func, **acq_params)
        
        queries[func_id] = next_query
        surrogates[func_id] = surrogate
        
        # Display
        best_x, best_y = func_data.get_best()
        mean, std = surrogate.predict(next_query.reshape(1, -1))
        
        print(f"Function {func_id} ({func_data.n_dims}D):")
        print(f"  Best so far: {best_y:.6f}")
        print(f"  Next query: {np.array2string(next_query, precision=6, separator=', ')}")
        print(f"  Predicted: mean={mean[0]:.6f}, std={std[0]:.6f}")
        print()
    
    print("=" * 80)
    return queries, surrogates

print("✓ Weekly query generator defined")

✓ Weekly query generator defined


## Generate Queries for This Week

## Advanced: Custom Strategy Per Function

This function allows you to specify different strategies for each function.

In [30]:
def generate_custom_queries(strategies: Dict[int, Dict] = None):
    """
    Generate queries with custom strategies per function.
    
    Args:
        strategies: Dict mapping function_id to strategy params
                   Example: {1: {'acq_func': 'ucb', 'beta': 2.0},
                            2: {'acq_func': 'ei', 'xi': 0.1}}
    
    If strategies is None, uses recommended defaults based on function characteristics.
    """
    # Default recommended strategies if none provided
    if strategies is None:
        strategies = {
            1: {'acq_func': 'ucb', 'beta': 2.5},  # 2D contamination - explore peaks
            2: {'acq_func': 'ei', 'xi': 0.01},    # Noisy - use EI
            3: {'acq_func': 'ucb', 'beta': 2.5},  # 3D drug - balanced
            4: {'acq_func': 'ucb', 'beta': 2.0},  # Dynamic - sustained exploration
            5: {'acq_func': 'ucb', 'beta': 2.0},  # Unimodal - can exploit more
            6: {'acq_func': 'ucb', 'beta': 2.5},  # 5D recipe - balanced
            7: {'acq_func': 'ucb', 'beta': 1.5},  # ML hyperparam - moderate
            8: {'acq_func': 'ucb', 'beta': 3.0},  # 8D - high exploration
        }
    
    queries = {}
    surrogates = {}
    
    print("=" * 80)
    print("GENERATING CUSTOM WEEKLY QUERIES")
    print("=" * 80)
    print()
    
    for func_id in range(1, 9):
        func_data = functions[func_id]
        strategy = strategies.get(func_id, {'acq_func': 'ucb', 'beta': 2.0})
        
        # Create and fit surrogate
        surrogate_params = strategy.get('surrogate_params', {'length_scale': 0.5, 'optimize': True})
        surrogate = GPSurrogate(**surrogate_params)
        surrogate.fit(func_data.inputs, func_data.outputs)
        
        # Extract acquisition params
        acq_func = strategy.get('acq_func', 'ucb')
        acq_params = {k: v for k, v in strategy.items() 
                     if k not in ['acq_func', 'surrogate_params']}
        
        # Find next query
        next_query = optimize_acquisition(surrogate, func_data, 
                                         acq_func=acq_func, **acq_params)
        
        queries[func_id] = next_query
        surrogates[func_id] = surrogate
        
        # Display
        best_x, best_y = func_data.get_best()
        mean, std = surrogate.predict(next_query.reshape(1, -1))
        
        print(f"Function {func_id} ({func_data.n_dims}D) - {acq_func.upper()} {acq_params}:")
        print(f"  Best so far: {best_y:.6f}")
        print(f"  Next query: {np.array2string(next_query, precision=6, separator=', ')}")
        print(f"  Predicted: mean={mean[0]:.6f}, std={std[0]:.6f}")
        print()
    
    print("=" * 80)
    return queries, surrogates

print("✓ Custom strategy query generator defined")

✓ Custom strategy query generator defined


## Option 1: Use Simple Generator (Same Strategy for All)

In [ ]:
# Simple approach: Same strategy for all functions
# weekly_queries, _ = generate_weekly_queries(acq_func='ucb', beta=2.0)

print("Uncomment above to use simple approach")

## Option 2: Use Custom Strategies (RECOMMENDED)

In [31]:
# Option A: Use recommended defaults (tuned for each function)
weekly_queries, weekly_surrogates = generate_custom_queries()

# Option B: Customize for specific functions
# my_strategies = {
#     1: {'acq_func': 'ucb', 'beta': 3.0},      # More exploration for function 1
#     2: {'acq_func': 'ei', 'xi': 0.1},         # EI for noisy function 2
#     5: {'acq_func': 'ei', 'xi': 0.01},        # Aggressive exploitation for unimodal
#     # Functions not specified use {'acq_func': 'ucb', 'beta': 2.0}
# }
# weekly_queries, weekly_surrogates = generate_custom_queries(my_strategies)

GENERATING CUSTOM WEEKLY QUERIES

Function 1 (2D) - UCB {'beta': 2.5}:
  Best so far: 0.000000
  Next query: [0.063271, 0.397595]
  Predicted: mean=-0.000313, std=0.001072

Function 2 (2D) - EI {'xi': 0.01}:
  Best so far: 0.611205
  Next query: [0.781541, 0.948752]
  Predicted: mean=0.636168, std=0.121992

Function 3 (3D) - UCB {'beta': 2.5}:
  Best so far: -0.034835
  Next query: [0.442633, 0.362694, 0.536069]
  Predicted: mean=-0.036940, std=0.078864

Function 4 (4D) - UCB {'beta': 2.0}:
  Best so far: -4.025542
  Next query: [0.413055, 0.408176, 0.347266, 0.431309]
  Predicted: mean=-1.140825, std=0.587126

Function 5 (4D) - UCB {'beta': 2.0}:
  Best so far: 1088.859618
  Next query: [0.378897, 0.842367, 0.958538, 0.999568]
  Predicted: mean=1092.128579, std=149.491462

Function 6 (5D) - UCB {'beta': 2.5}:
  Best so far: -0.714265
  Next query: [0.248237, 0.314846, 0.411998, 0.758207, 0.      ]
  Predicted: mean=-0.664995, std=0.368419

Function 7 (6D) - UCB {'beta': 1.5}:
  Best s

## IMPORTANT: Portal Submission Format

**The portal expects HYPHENS, not commas!**  
Format: `x1-x2-x3-...-xn`

In [ ]:
def format_for_portal(queries: Dict[int, np.ndarray]):
    """
    Format queries correctly for the competition portal.
    Portal expects: x1-x2-x3-...-xn (HYPHENS not commas!)
    """
    print("=" * 80)
    print("PORTAL SUBMISSION FORMAT")
    print("=" * 80)
    print()
    
    for func_id in range(1, 9):
        if func_id in queries:
            query = queries[func_id]
            # Portal format uses HYPHENS between values
            query_str = "-".join([f"{x:.6f}" for x in query])
            print(f"Function {func_id}: {query_str}")
        else:
            print(f"Function {func_id}: -")
    
    print()
    print("=" * 80)
    print("Copy the values above and paste into the portal")
    print("=" * 80)

# Format with correct portal format (hyphens!)
format_for_portal(weekly_queries)

In [14]:
# Generate queries using UCB acquisition function
weekly_queries, weekly_surrogates = generate_weekly_queries(acq_func='ucb', beta=2.0)

GENERATING WEEKLY QUERIES FOR ALL FUNCTIONS
Acquisition function: UCB
Acquisition params: {'beta': 2.0}

Function 1 (2D):
  Best so far: 0.000000
  Next query: [0.0991  , 0.412748]
  Predicted: mean=-0.000301, std=0.001067

Function 2 (2D):
  Best so far: 0.611205
  Next query: [0.816323, 0.980338]
  Predicted: mean=0.567152, std=0.174584

Function 3 (3D):
  Best so far: -0.034835
  Next query: [0.42674 , 0.378188, 0.522189]
  Predicted: mean=-0.029781, std=0.075669

Function 4 (4D):
  Best so far: -4.025542
  Next query: [0.413055, 0.408176, 0.347266, 0.431309]
  Predicted: mean=-1.140825, std=0.587126

Function 5 (4D):
  Best so far: 1088.859618
  Next query: [0.378897, 0.842367, 0.958538, 0.999568]
  Predicted: mean=1092.128579, std=149.491462

Function 6 (5D):
  Best so far: -0.714265
  Next query: [0.285591, 0.32769 , 0.429042, 0.754276, 0.027606]
  Predicted: mean=-0.606588, std=0.342056

Function 7 (6D):
  Best so far: 1.364968
  Next query: [0.028118, 0.402838, 0.322036, 0.1654

## Format Queries for Submission

In [32]:
def format_queries_for_submission(queries: Dict[int, np.ndarray]):
    """Format queries in a submission-ready format."""
    print("=" * 80)
    print("WEEKLY SUBMISSION - QUERY POINTS")
    print("=" * 80)
    print()
    
    for func_id, query in queries.items():
        print(f"Function {func_id}:")
        query_str = ", ".join([f"{x:.6f}" for x in query])
        print(f"  {query_str}")
        print()
    
    print("=" * 80)

# Format the generated queries
format_queries_for_submission(weekly_queries)

WEEKLY SUBMISSION - QUERY POINTS

Function 1:
  0.063271, 0.397595

Function 2:
  0.781541, 0.948752

Function 3:
  0.442633, 0.362694, 0.536069

Function 4:
  0.413055, 0.408176, 0.347266, 0.431309

Function 5:
  0.378897, 0.842367, 0.958538, 0.999568

Function 6:
  0.248237, 0.314846, 0.411998, 0.758207, 0.000000

Function 7:
  0.026971, 0.427861, 0.300063, 0.167708, 0.357189, 0.730492

Function 8:
  0.138513, 0.000000, 0.000000, 0.000000, 1.000000, 0.000000, 0.000000, 0.000000



# Section 6: Update Models with New Results

When you receive results from weekly submissions, use these cells to update your models.

In [ ]:
def update_function_with_result(func_id: int, x: np.ndarray, y: float, 
                                week: Optional[int] = None, save: bool = True):
    """Update a function with a new observation."""
    func_data = functions[func_id]
    func_data.add_observation(x, y, week)
    
    if save and week is not None:
        func_data.save_weekly_data(week)
    
    best_x, best_y = func_data.get_best()
    print(f"✓ Function {func_id} updated:")
    print(f"  New observation: y = {y:.6f}")
    print(f"  Best so far: {best_y:.6f}")
    print(f"  Total samples: {func_data.n_samples}")

print("✓ Update function defined")

## Example: Update Functions with Results

Uncomment and modify these lines when you receive results.

In [ ]:
# Example: Update function 1 with result from week 1
# x_new = weekly_queries[1]  # The query you submitted
# y_new = 0.123456  # The result you received
# update_function_with_result(1, x_new, y_new, week=1, save=True)

# Update all functions at once:
# for func_id in range(1, 9):
#     x_new = weekly_queries[func_id]
#     y_new = ... # Replace with actual result
#     update_function_with_result(func_id, x_new, y_new, week=1, save=True)

print("Update template ready - modify and uncomment when results arrive")

# Section 7: Progress Tracking

Monitor optimization progress over time.

In [ ]:
def plot_progress(func_ids: List[int] = None):
    """Plot optimization progress for selected functions."""
    if func_ids is None:
        func_ids = list(range(1, 9))
    
    n_funcs = len(func_ids)
    n_cols = min(3, n_funcs)
    n_rows = (n_funcs + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 4*n_rows))
    if n_funcs == 1:
        axes = [axes]
    else:
        axes = axes.flatten() if n_funcs > 1 else [axes]
    
    for idx, func_id in enumerate(func_ids):
        func_data = functions[func_id]
        ax = axes[idx]
        
        # Plot cumulative best
        cumulative_best = np.maximum.accumulate(func_data.outputs)
        samples = np.arange(1, len(cumulative_best) + 1)
        
        ax.plot(samples, cumulative_best, 'b-', linewidth=2, label='Best Value')
        ax.scatter(samples, func_data.outputs, c='lightblue', 
                  alpha=0.5, s=30, label='Observations')
        
        # Highlight weekly updates if any
        if func_data.history:
            history_values = [h[2] for h in func_data.history]
            history_indices = list(range(len(func_data.outputs) - len(history_values), 
                                       len(func_data.outputs)))
            ax.scatter(np.array(history_indices) + 1, history_values, 
                      c='red', s=100, marker='*', 
                      label='Weekly Updates', zorder=5)
        
        ax.set_xlabel('Sample Number')
        ax.set_ylabel('Function Value')
        ax.set_title(f'Function {func_id} ({func_data.n_dims}D)')
        ax.legend(loc='best')
        ax.grid(True, alpha=0.3)
    
    # Hide empty subplots
    for idx in range(n_funcs, len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.show()

print("✓ Progress plotting function defined")

## View Progress for All Functions

In [ ]:
# Plot progress for all functions
plot_progress()

## Summary Statistics

In [ ]:
def display_competition_summary():
    """Display overall competition progress."""
    print("=" * 80)
    print("COMPETITION SUMMARY")
    print("=" * 80)
    print()
    
    total_queries = sum(f.week_number for f in functions.values())
    
    print(f"Total weekly submissions: {max(f.week_number for f in functions.values())}")
    print(f"Total queries made: {total_queries}")
    print()
    print("Best values by function:")
    print("-" * 80)
    
    for func_id in range(1, 9):
        func_data = functions[func_id]
        best_x, best_y = func_data.get_best()
        improvement = best_y - func_data.outputs[0] if len(func_data.outputs) > 0 else 0
        
        print(f"Function {func_id} ({func_data.n_dims}D): {best_y:.6f} "
              f"(+{improvement:.6f} improvement, {func_data.n_samples} samples)")
    
    print("=" * 80)

display_competition_summary()

: 

## Format Queries for Submission

Display queries in a format ready for copy-paste submission.

In [ ]:
def format_queries_for_submission(queries: Dict[int, np.ndarray]):
    """Format queries in a submission-ready format."""
    print("=" * 80)
    print("WEEKLY SUBMISSION - QUERY POINTS")
    print("=" * 80)
    print()
    
    for func_id, query in queries.items():
        print(f"Function {func_id}:")
        # Format as comma-separated values for easy copy-paste
        query_str = ", ".join([f"{x:.6f}" for x in query])
        print(f"  {query_str}")
        print()
    
    print("=" * 80)

# Format the generated queries
format_queries_for_submission(weekly_queries)

# Section 6: Update Models with New Results

Add new observations when you receive results from the weekly submission.

In [ ]:
def update_function_with_result(func_id: int, x: np.ndarray, y: float, 
                                week: Optional[int] = None, save: bool = True):
    """
    Update a function with a new observation.
    
    Args:
        func_id: Function ID (1-8)
        x: Input point
        y: Observed output
        week: Week number (auto-increments if None)
        save: Whether to save updated data to disk
    """
    func_data = functions[func_id]
    func_data.add_observation(x, y, week)
    
    if save and week is not None:
        func_data.save_weekly_data(week)
    
    best_x, best_y = func_data.get_best()
    print(f"✓ Function {func_id} updated:")
    print(f"  New observation: y = {y:.6f}")
    print(f"  Best so far: {best_y:.6f}")
    print(f"  Total samples: {func_data.n_samples}")

print("✓ Update function defined")

## Example: Update Function with New Result

When you receive results, update each function using this template.

In [ ]:
# Example: Update function 1 with result from week 1
# Replace with actual query point and result
# x_new = weekly_queries[1]  # The query you submitted
# y_new = 0.123456  # The result you received
# update_function_with_result(1, x_new, y_new, week=1, save=True)

print("Example code ready - uncomment and fill in actual values when results arrive")

# Section 7: Progress Tracking

Track optimization progress over time.

In [ ]:
def plot_progress(func_ids: List[int] = None):
    """
    Plot optimization progress for selected functions.
    
    Args:
        func_ids: List of function IDs to plot (None = all)
    """
    if func_ids is None:
        func_ids = list(range(1, 9))
    
    n_funcs = len(func_ids)
    n_cols = min(3, n_funcs)
    n_rows = (n_funcs + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 4*n_rows))
    if n_funcs == 1:
        axes = [axes]
    else:
        axes = axes.flatten() if n_funcs > 1 else [axes]
    
    for idx, func_id in enumerate(func_ids):
        func_data = functions[func_id]
        ax = axes[idx]
        
        # Plot cumulative best
        cumulative_best = np.maximum.accumulate(func_data.outputs)
        samples = np.arange(1, len(cumulative_best) + 1)
        
        ax.plot(samples, cumulative_best, 'b-', linewidth=2, label='Best Value')
        ax.scatter(samples, func_data.outputs, c='lightblue', 
                  alpha=0.5, s=30, label='Observations')
        
        # Highlight new observations from history
        if func_data.history:
            history_weeks = [h[0] for h in func_data.history]
            history_values = [h[2] for h in func_data.history]
            history_indices = list(range(len(func_data.outputs) - len(history_values), 
                                       len(func_data.outputs)))
            ax.scatter(np.array(history_indices) + 1, history_values, 
                      c='red', s=100, marker='*', 
                      edgecolors='darkred', linewidths=1,
                      label='Weekly Updates', zorder=5)
        
        ax.set_xlabel('Sample Number')
        ax.set_ylabel('Function Value')
        ax.set_title(f'Function {func_id} ({func_data.n_dims}D) - Progress')
        ax.legend(loc='best')
        ax.grid(True, alpha=0.3)
    
    # Hide empty subplots
    for idx in range(n_funcs, len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.show()

print("✓ Progress plotting function defined")

## View Progress for All Functions

In [ ]:
# Plot progress for all functions
plot_progress()

## Summary Statistics Across All Functions

In [ ]:
def display_competition_summary():
    """Display overall competition progress."""
    print("=" * 80)
    print("COMPETITION SUMMARY")
    print("=" * 80)
    print()
    
    total_queries = sum(f.week_number for f in functions.values())
    
    print(f"Total weekly submissions: {max(f.week_number for f in functions.values())}")
    print(f"Total queries made: {total_queries}")
    print()
    print("Best values by function:")
    print("-" * 80)
    
    for func_id in range(1, 9):
        func_data = functions[func_id]
        best_x, best_y = func_data.get_best()
        improvement = best_y - func_data.outputs[0] if len(func_data.outputs) > 0 else 0
        
        print(f"Function {func_id} ({func_data.n_dims}D): {best_y:.6f} "
              f"(+{improvement:.6f} improvement, {func_data.n_samples} samples)")
    
    print("=" * 80)

display_competition_summary()